# Notebook 12 — CHIRPS sobre cuenca alta del Magdalena y Sierra Nevada como proxy del forzante fluvial

Cierra la cadena causal La Niña → caudal fluvial → mortandad del manglar mediante un proxy global: la precipitación CHIRPS (Climate Hazards Group InfraRed Precipitation with Station data, Funk et al. 2015) sobre las dos cuencas que aportan agua al sistema CGSM ---la cuenca alta-media del río Magdalena (desde Huila hasta el bajo Magdalena) y la vertiente occidental de la Sierra Nevada de Santa Marta (tributarios Sevilla, Aracataca, Fundación)---. El uso de CHIRPS como sustituto del caudal IDEAM ---no disponible automáticamente al momento de la entrega--- se justifica en la medida en que CHIRPS incorpora estaciones meteorológicas colombianas en su calibración, ofrece serie continua desde 1981 a 0,05° de resolución, y captura la dinámica de precipitación sobre las áreas aportantes que media entre el forzamiento ENSO y la inundación del humedal.

**Insumos:**
- GEE: `UCSB-CHG/CHIRPS/DAILY` (CHIRPS v2.0)
- `outputs/tables/serie_temporal_ndvi_definitiva.csv`

**Productos:**
- `outputs/tables/chirps_cuencas_mensual.csv`
- `outputs/tables/correlacion_chirps_ndvi.csv`
- `outputs/figures/chirps_serie_cuencas_2013_2025.png`
- `outputs/figures/chirps_vs_ndvi_correlacion.png`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ee

try:
    ee.Initialize(project='basic-buttress-338101')
except Exception:
    import google.auth
    creds, _ = google.auth.default()
    ee.Initialize(credentials=creds, project='basic-buttress-338101')

ROOT = Path('..').resolve()
OUT_TAB = ROOT / 'outputs' / 'tables'
OUT_FIG = ROOT / 'outputs' / 'figures'
OUT_TAB.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

## 1. Definir las dos cuencas aportantes

La cuenca alta-media del río Magdalena cubre la cordillera Andina central colombiana entre 2° y 9° de latitud norte. La vertiente noroccidental de la Sierra Nevada drena directamente a los tributarios Sevilla, Aracataca y Fundación que entran al sistema lagunar de la CGSM.

In [ ]:
# Cuenca alta-media del Magdalena: corredor entre Huila y bajo Magdalena
cuenca_magdalena = ee.Geometry.Rectangle([-76.0, 2.0, -74.0, 9.0])

# Sierra Nevada de Santa Marta (vertiente oeste, drena a tributarios CGSM)
sierra_nevada = ee.Geometry.Rectangle([-74.2, 10.4, -73.4, 11.3])

cuencas = {
    'magdalena_alta_media': cuenca_magdalena,
    'sierra_nevada_oeste': sierra_nevada,
}

print('Cuencas definidas:')
for nombre, geom in cuencas.items():
    area_km2 = geom.area().divide(1e6).getInfo()
    print(f'  {nombre:25s} ~ {area_km2:,.0f} km²')

## 2. Extraer serie temporal mensual de CHIRPS sobre cada cuenca

CHIRPS diario se agrega a totales mensuales y se promedia espacialmente sobre cada cuenca en una sola llamada a Earth Engine usando `ImageCollection.map(reduceRegion)` para máxima eficiencia.

In [ ]:
chirps_daily = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                .filterDate('2013-01-01', '2025-12-31')
                .select('precipitation'))

# Construir lista de meses
years = list(range(2013, 2026))
months = list(range(1, 13))
ym_list = [(y, m) for y in years for m in months]

def imagen_mensual(y, m):
    start = ee.Date.fromYMD(y, m, 1)
    end = start.advance(1, 'month')
    return chirps_daily.filterDate(start, end).sum().set('date', start.format('YYYY-MM-15'))

imgs_mensuales = ee.ImageCollection([imagen_mensual(y, m) for y, m in ym_list])
print(f'Imágenes mensuales construidas: {imgs_mensuales.size().getInfo()}')

# Para cada cuenca, extraer la serie con una sola llamada
series = {}
for nombre, geom in cuencas.items():
    print(f'\nExtrayendo serie sobre {nombre}...')
    fc = imgs_mensuales.map(lambda img: ee.Feature(None, {
        'date': img.get('date'),
        'precip_mm': img.reduceRegion(ee.Reducer.mean(), geom, 5000).get('precipitation')
    }))
    data = fc.getInfo()['features']
    df = pd.DataFrame([f['properties'] for f in data])
    df['date'] = pd.to_datetime(df['date'])
    df['cuenca'] = nombre
    series[nombre] = df.sort_values('date').reset_index(drop=True)
    print(f'  {len(df)} meses, precip media {df.precip_mm.mean():.1f} mm/mes')

chirps_df = pd.concat(series.values(), ignore_index=True)
chirps_df.to_csv(OUT_TAB / 'chirps_cuencas_mensual.csv', index=False)
print(f'\nGuardado: chirps_cuencas_mensual.csv ({len(chirps_df)} filas)')

## 3. Calcular anomalías estandarizadas (z-score) por cuenca

Se remueve la climatología mensual de cada cuenca y se divide por la desviación estándar para obtener un z-score comparable con la anomalía NDVI.

In [ ]:
def anomalia_z(df):
    df = df.copy()
    df['mes'] = df['date'].dt.month
    clim = df.groupby('mes')['precip_mm'].agg(['mean', 'std']).reset_index()
    df = df.merge(clim, on='mes', how='left')
    df['precip_z'] = (df['precip_mm'] - df['mean']) / df['std']
    return df[['date', 'cuenca', 'precip_mm', 'precip_z']]

chirps_anom = pd.concat([anomalia_z(series[c]) for c in cuencas], ignore_index=True)
print(chirps_anom.groupby('cuenca')['precip_z'].describe())

## 4. Visualización temporal con eventos ENSO destacados

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for ax, cuenca in zip(axes, cuencas.keys()):
    sub = chirps_anom[chirps_anom.cuenca == cuenca]
    colors = ['#D32F2F' if z > 0 else '#1976D2' for z in sub.precip_z]
    ax.bar(sub.date, sub.precip_z, color=colors, alpha=0.6, width=25)
    ax.axhline(0, color='black', lw=0.5)
    ax.axvspan(pd.Timestamp('2015-06-01'), pd.Timestamp('2016-05-31'),
               alpha=0.08, color='red')
    ax.axvspan(pd.Timestamp('2020-08-01'), pd.Timestamp('2022-03-31'),
               alpha=0.08, color='blue')
    ax.set_ylabel('z-score precipitación')
    ax.set_title(f'CHIRPS — {cuenca.replace("_", " ")}', fontsize=11)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Fecha')
fig.suptitle('Anomalías mensuales CHIRPS sobre cuencas aportantes a la CGSM, 2013–2025', y=1.01)
plt.tight_layout()
plt.savefig(OUT_FIG / 'chirps_serie_cuencas_2013_2025.png', dpi=200, bbox_inches='tight')
plt.show()
print('Guardado: chirps_serie_cuencas_2013_2025.png')

## 5. Correlación CHIRPS vs NDVI z-score desagregada por naturaleza espectral

In [ ]:
NDVI_CSV = OUT_TAB / 'serie_temporal_ndvi_definitiva.csv'
ndvi = pd.read_csv(NDVI_CSV, parse_dates=['date'])
ndvi['z'] = ndvi.groupby('subzona')['ndvi'].transform(
    lambda x: (x - x.mean()) / x.std())
ndvi['date_m'] = ndvi['date'].dt.to_period('M').dt.to_timestamp() + pd.offsets.Day(14)

manglar      = {'Cano_Palos', 'Cano_Clarin', 'CP_Aguas_Negras', 'CP_Luna'}
limnologica  = {'Isla_Boqueron', 'Punta_Cerro', 'Punta_Chino', 'Rio_Sevilla'}
ndvi['naturaleza'] = ndvi['subzona'].apply(
    lambda s: 'manglar' if s in manglar else ('limnologica' if s in limnologica else 'otra'))

z_mensual = ndvi.groupby(['date_m', 'naturaleza'])['z'].mean().reset_index()
z_mensual = z_mensual.rename(columns={'date_m': 'date'})

filas = []
for cuenca in cuencas.keys():
    serie_p = chirps_anom[chirps_anom.cuenca == cuenca][['date', 'precip_z']]
    for nat in ['manglar', 'limnologica']:
        serie_n = z_mensual[z_mensual.naturaleza == nat]
        merged = serie_n.merge(serie_p, on='date', how='inner').sort_values('date')
        for lag in range(0, 4):
            rho = merged['z'].corr(merged['precip_z'].shift(lag))
            n = (~merged['precip_z'].shift(lag).isna() & ~merged['z'].isna()).sum()
            filas.append({
                'cuenca': cuenca, 'naturaleza': nat, 'rezago_meses': lag,
                'rho': round(rho, 3) if pd.notna(rho) else None, 'n': int(n),
            })

df_corr = pd.DataFrame(filas)
df_corr.to_csv(OUT_TAB / 'correlacion_chirps_ndvi.csv', index=False)
print(df_corr.to_string(index=False))
print(f'\nGuardado: correlacion_chirps_ndvi.csv')

## 6. Figura comparativa de correlaciones por rezago, por cuenca y naturaleza

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True, sharex=True)
for i, cuenca in enumerate(cuencas.keys()):
    for j, nat in enumerate(['manglar', 'limnologica']):
        ax = axes[i, j]
        sub = df_corr[(df_corr.cuenca == cuenca) & (df_corr.naturaleza == nat)]
        colors = ['#D32F2F' if r < 0 else '#1976D2' for r in sub['rho']]
        ax.bar(sub.rezago_meses, sub['rho'], color=colors, alpha=0.7)
        ax.axhline(0, color='black', lw=0.5)
        ax.set_title(f'{cuenca.replace("_", " ")} → {nat}', fontsize=10)
        ax.set_xticks(range(4))
        ax.grid(axis='y', alpha=0.3)
        if j == 0: ax.set_ylabel(r'$\rho$ Pearson')
        if i == 1: ax.set_xlabel('Rezago (meses)')

fig.suptitle('Correlación entre precipitación CHIRPS sobre cuencas aportantes y NDVI z-score de la CGSM, 2013–2025',
             y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig(OUT_FIG / 'chirps_vs_ndvi_correlacion.png', dpi=200, bbox_inches='tight')
plt.show()
print('Guardado: chirps_vs_ndvi_correlacion.png')

## 7. Interpretación esperada

La precipitación CHIRPS sobre la cuenca alta-media del Magdalena debería preceder al estrés del manglar con rezagos del orden de 1 a 3 meses, en la medida en que el agua que cae en la cordillera tarda ese lapso en llegar al sistema lagunar a través del río Magdalena. La cuenca de la Sierra Nevada, en cambio, podría mostrar un acoplamiento más rápido pues los tributarios menores (Sevilla, Aracataca, Fundación) tienen tiempos de tránsito de horas a pocos días, por lo que la respuesta en NDVI debería aparecer con rezago de 0 a 1 mes. Si la correlación con cuenca alta-media Magdalena resulta más fuerte que la observada con ERA5-Land local ---ver tabla `tbl-era5`---, queda confirmado que el forzamiento hídrico operativo sobre el manglar de la CGSM no es la lluvia caída sobre el humedal sino el caudal aportado por las cuencas altas, lo que cierra la cadena causal La Niña → precipitación cuenca alta → caudal río → inundación humedal → mortandad manglar planteada en el informe.